Aim of this script: build the final operator / route-type / year summary from nb4's per-train (`journeys`) table, and export it as an Excel workbook in `summary_stats/`.

In [1]:
import pandas as pd
import os

data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
journeys_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_journeys.parquet"

try:
    journeys.head()
except NameError:
    journeys = pd.read_parquet(journeys_path)

journeys.shape

(405471, 17)

### Aggregating to Year / operator / Route type

Each row of `journeys` (from nb4) is already one train. `operator` is null wherever nb3 couldn't resolve one -- rather than silently dropping those trains from the groupby, they're kept under an explicit `UNKNOWN_OPERATOR` label so the totals in the summary still add up to the full `journeys` count.

Column definitions (see nb4 for how each is derived):
- **N total train** -- every train in scope, regardless of cancellation/delay/operator status
- **N cancelled train (full cancellation)** -- every stop of the train was cancelled
- **N cancelled train (partial cancellation)** -- some, but not all, stops were cancelled
- **N cancelled train (full + partial)** -- either of the above
- **N train delay (> 5 min) / (> 15 min)** -- arrived at the terminus, but later than the threshold
- **N train arriving at terminus** -- the terminus stop was reached (not cancelled), independent of delay

In [2]:
journeys["operator"] = journeys["operator"].fillna("UNKNOWN_OPERATOR")
print(f"{(journeys['operator'] == 'UNKNOWN_OPERATOR').mean() * 100:.2f}% of trains have no resolved operator")

summary = journeys.groupby(["year", "operator", "routeType"], dropna=False).agg(
    n_total_train=("journey_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    n_delay_5min=("delayed_5min", "sum"),
    n_delay_15min=("delayed_15min", "sum"),
    n_arriving_terminus=("arrived_at_terminus", "sum"),
).reset_index()

summary["n_cancelled_full_or_partial"] = summary["n_cancelled_full"] + summary["n_cancelled_partial"]

assert summary["n_total_train"].sum() == len(journeys), "grouped total doesn't match the journey count -- a group key must be dropping nulls"

summary.head()

4.16% of trains have no resolved operator


,year,operator,routeType,n_total_train,n_cancelled_full,n_cancelled_partial,n_delay_5min,n_delay_15min,n_arriving_terminus,n_cancelled_full_or_partial
0,2025,Deutsche Bahn,ICE,6760,0,0,3148,1566,6760,0
1,2025,Eurostar,EST,42687,1775,2646,10547,6310,40397,4421
2,2025,SNCF,INTERCITES,18397,0,0,4136,2697,18397,0
3,2025,SNCF,INTERCITES DE NUIT,2394,0,0,175,123,2394,0
4,2025,SNCF,LYRIA,8973,0,0,1520,842,8973,0


### Exporting the summary as Excel

In [3]:
summary = summary.rename(columns={
    "year": "Year",
    "routeType": "Route type",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    "n_delay_5min": "N train delay (> 5 min)",
    "n_delay_15min": "N train delay (> 15 min)",
    "n_arriving_terminus": "N train arriving at terminus",
})

column_order = [
    "Year", "operator", "Route type",
    "N total train",
    "N cancelled train (full cancellation)",
    "N cancelled train (partial cancellation)",
    "N cancelled train (full + partial)",
    "N train delay (> 5 min)",
    "N train delay (> 15 min)",
    "N train arriving at terminus",
]
summary = summary[column_order].sort_values(["Year", "operator", "Route type"])
summary

,Year,operator,Route type,N total train,N cancelled train (full cancellation),N cancelled train (partial cancellation),N cancelled train (full + partial),N train delay (> 5 min),N train delay (> 15 min),N train arriving at terminus
0,2025,Deutsche Bahn,ICE,6760,0,0,0,3148,1566,6760
1,2025,Eurostar,EST,42687,1775,2646,4421,10547,6310,40397
2,2025,SNCF,INTERCITES,18397,0,0,0,4136,2697,18397
3,2025,SNCF,INTERCITES DE NUIT,2394,0,0,0,175,123,2394
4,2025,SNCF,LYRIA,8973,0,0,0,1520,842,8973
5,2025,SNCF,NAVETTE,47,0,0,0,3,1,47
6,2025,SNCF,OUIGO,16164,0,0,0,3290,2015,16164
7,2025,SNCF,TGV INOUI,139731,0,0,0,24885,13640,139728
8,2025,SNCF,TRAIN TER,148494,0,0,0,11471,5665,148486
9,2025,Trenitalia,FR,4944,0,0,0,1104,494,4943


In [4]:
export_summary = input("Export summary to Excel? (y/n): ")

if export_summary.lower() == "y":
    summary_stats_dir = "summary_stats"
    os.makedirs(summary_stats_dir, exist_ok=True)

    summary_path = f"{summary_stats_dir}/chuuchuu_summary_{data_selection}.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Saved to {summary_path}")

Saved to summary_stats/chuuchuu_summary_french.xlsx